# Notebook 1 — Softmax, cross-entropy e Hessiana multiclasse

**Concetto (dispense §7.3).** Passiamo dalla Laplace 1-D al caso che ci serve davvero:
la classificazione **multiclasse** con softmax. Dalle dispense:
$$p(C_k\mid x)=\sigma_k\!\big(W^{\!\top}\phi(x)\big),\qquad
\sigma_k(a)=\frac{e^{a_k}}{\sum_j e^{a_j}},$$
con cross-entropy
$$E(w)=-\frac1N\sum_{n}\sum_{k} y_{nk}\log s_{nk},\qquad s_{nk}=\sigma_k(W^{\!\top}\phi(x_n)),$$
e gradiente (dispense) $\;\nabla_{w_j}E=\frac1N\sum_n (s_{nj}-y_{nj})\,\phi(x_n)$.

Per la Laplace ci serve **l'Hessiana**. Il fatto chiave di questo notebook è che
l'Hessiana della cross-entropy **rispetto ai logit** è
$$\boxed{\,\Lambda=\operatorname{diag}(p)-p\,p^{\!\top}\,}$$
— e per un modello **lineare** (logit $=W\phi$) questa è *esatta*, non un'approssimazione.

**Prova di comprensione (obiettivo).**
(1) derivare/verificare $\Lambda=\operatorname{diag}(p)-pp^{\!\top}$ e assemblare
l'Hessiana nello spazio dei pesi $\;H=\sum_n \Lambda_n\otimes\phi_n\phi_n^{\!\top}$,
confrontandola con l'Hessiana **numerica** della CE;
(2) mostrare che per $K=2$ si riduce alla logistica binaria delle dispense.

In [1]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")  # evita OMP Error #15 (Windows)

import sys, pathlib
# root del progetto = cartella che contiene 'src', trovata a partire dal notebook
here = pathlib.Path.cwd()
for base in [here, *here.parents]:
    if (base / "src").is_dir():
        sys.path.insert(0, str(base)); PROJ = base; break
else:
    raise RuntimeError("cartella 'src' non trovata: apri il progetto dalla sua root")
import numpy as np

from src.laplace_basics import (
    softmax, softmax_hessian_block, weight_space_hessian,
    numeric_hessian, cross_entropy_sum,
)
np.set_printoptions(precision=4, suppress=True)

## 1. Il blocco $\Lambda=\operatorname{diag}(p)-pp^{\top}$

Per un singolo punto con probabilità $p=\sigma(a)$, la derivata della softmax è
$\dfrac{\partial p_k}{\partial a_l}=p_k(\delta_{kl}-p_l)$. Componendola con la
cross-entropy si ottiene che l'Hessiana della CE rispetto ai logit $a$ è
$\Lambda=\operatorname{diag}(p)-pp^{\top}$.

Due proprietà che la caratterizzano e che verifichiamo: è **simmetrica** e ha
**righe/colonne a somma nulla** (perché i logit sono definiti a meno di una costante
comune → esiste una direzione "di gauge" a curvatura zero).

In [2]:
p = np.array([0.6, 0.3, 0.1])
Lam = softmax_hessian_block(p)
print("Λ =\n", Lam)
print("\nsimmetrica      :", np.allclose(Lam, Lam.T))
print("righe a somma 0 :", np.allclose(Lam.sum(axis=1), 0))
print("autovalori      :", np.linalg.eigvalsh(Lam), " (uno è ~0: la direzione di gauge)")

Λ =
 [[ 0.24 -0.18 -0.06]
 [-0.18  0.21 -0.03]
 [-0.06 -0.03  0.09]]

simmetrica      : True
righe a somma 0 : True
autovalori      : [-0.      0.1325  0.4075]  (uno è ~0: la direzione di gauge)


## 2. Hessiana nello spazio dei pesi

Con logit $a_n=W\phi_n$ e regola della catena, l'Hessiana della CE **sommata** sui
punti rispetto ai pesi è
$$H=\sum_{n}\Lambda_n\otimes\big(\phi_n\phi_n^{\!\top}\big),\qquad
\Lambda_n=\operatorname{diag}(p_n)-p_np_n^{\!\top}.$$
Aggiungendo un prior gaussiano $\mathcal N(0,\tau^{-1}I)$ sui pesi compare il termine
$+\,\tau I$ — che sarà il ponte verso la posterior $\mathcal N(\theta_{\text{MAP}},H^{-1})$.

Costruiamo un problemino lineare (K=3, D=2) e assembliamo $H$.

In [3]:
rng = np.random.default_rng(0)
N, D, K = 12, 2, 3
Phi = rng.normal(size=(N, D))          # feature (N, D)
y   = rng.integers(0, K, size=N)       # etichette
W   = rng.normal(size=(K, D)) * 0.5    # pesi (K, D)

H = weight_space_hessian(Phi, W, tau=0.0)
print("H shape:", H.shape, " (K*D x K*D =", K*D, "x", K*D, ")")

H shape: (6, 6)  (K*D x K*D = 6 x 6 )


### Verifica: Hessiana analitica $=$ Hessiana numerica

Se la derivazione è corretta, $H=\sum_n\Lambda_n\otimes\phi_n\phi_n^{\top}$ deve
coincidere con l'Hessiana calcolata **per differenze finite** direttamente sulla
cross-entropy sommata. La coincidenza (a meno del rumore numerico) è la prova che
per l'ultimo layer lineare **non c'è alcuna approssimazione**: la Gauss-Newton *è*
l'Hessiana esatta.

In [4]:
w_flat = W.reshape(-1)
H_num = numeric_hessian(lambda w: cross_entropy_sum(w, Phi, y, K), w_flat, h=1e-4)

err = np.abs(H - H_num).max()
print(f"‖H_analitica − H_numerica‖_max = {err:.3e}")
assert err < 1e-4, "la derivazione dell'Hessiana non torna"
print("OK — coincidono: per un layer lineare la GGN è l'Hessiana esatta.")

‖H_analitica − H_numerica‖_max = 1.339e-07
OK — coincidono: per un layer lineare la GGN è l'Hessiana esatta.


## 3. Riduzione a $K=2$ → logistica binaria (dispense)

Le dispense danno l'Hessiana della **logistica binaria** come
$\sum_n s_n(1-s_n)\,\phi_n\phi_n^{\top}$. Mostriamo che è il caso particolare $K=2$
del nostro $\Lambda$. Con $p=[s,\,1-s]$:
$$\Lambda=\begin{pmatrix} s(1-s) & -s(1-s)\\ -s(1-s) & s(1-s)\end{pmatrix}
= s(1-s)\begin{pmatrix}1&-1\\-1&1\end{pmatrix}.$$
Il modello a 2 classi è **sovra-parametrizzato** (conta solo la differenza dei logit):
$\Lambda$ ha un autovalore $0$ (la gauge) e uno $2s(1-s)$. Fissando la classe 2 come
riferimento ($w_2=0$), la curvatura residua è l'elemento $\Lambda_{11}=s(1-s)$ —
**esattamente** il fattore della logistica binaria.

In [5]:
s = 0.3
Lam2 = softmax_hessian_block([s, 1 - s])
print("Λ(K=2) =\n", Lam2)
print("\n= s(1-s)·[[1,-1],[-1,1]] ?", np.allclose(Lam2, s*(1-s)*np.array([[1,-1],[-1,1]])))
print("autovalori:", np.linalg.eigvalsh(Lam2), f" (gauge=0, 2s(1-s)={2*s*(1-s):.4f})")
print(f"curvatura residua fissando la classe 2 = Λ[0,0] = s(1-s) = {s*(1-s):.4f}")
print("  -> è l'Hessiana della logistica binaria delle dispense.")

Λ(K=2) =
 [[ 0.21 -0.21]
 [-0.21  0.21]]

= s(1-s)·[[1,-1],[-1,1]] ? True
autovalori: [0.   0.42]  (gauge=0, 2s(1-s)=0.4200)
curvatura residua fissando la classe 2 = Λ[0,0] = s(1-s) = 0.2100
  -> è l'Hessiana della logistica binaria delle dispense.


## Riassunto

- L'Hessiana della cross-entropy nei logit è $\Lambda=\operatorname{diag}(p)-pp^{\top}$, **esatta**.
- Nello spazio dei pesi (layer lineare): $H=\sum_n\Lambda_n\otimes\phi_n\phi_n^{\top}+\tau I$,
  confermata contro l'Hessiana numerica.
- Per $K=2$ ritorna la logistica binaria delle dispense.

Questi sono esattamente i mattoni della **last-layer Laplace**: nel Notebook 2 il
feature extractor $g$ (congelato) produce le $\phi_n$, e su questa $H$ costruiamo la
posterior $\mathcal N(\theta_{\text{MAP}},H^{-1})$ sull'ultimo layer del modello che abbiamo già.